In [ ]:
import os
import requests
import json
from os.path import expanduser
from requests.auth import HTTPBasicAuth

# 加载凭据文件
current_dir = os.getcwd()
credentials_path = os.path.join(current_dir, 'brain_credentials.txt')

# 使用正确的文件路径
with open(credentials_path) as f:
    credentials = json.load(f)

# 从凭据列表中提取用户名和密码
username, password = credentials

# 创建会话对象
sess = requests.Session()

# 设置基本认证
sess.auth = HTTPBasicAuth(username, password)

# 发送认证请求
response = sess.post('https://api.worldquantbrain.com/authentication')

# 打印响应状态和内容（使用英文括号）
print(response.status_code)
print(response.json())

In [ ]:
# 获取数据集ID为fundamental6（Company Fundamental Data for Equity）下的所有数据字段
### Get Data_fields like Data Explorer 获取所有满足条件的数据字段及其ID
def get_datafields(
        s,
        searchScope,
        dataset_id: str = '',
        search: str = ''
):
    import pandas as pd
    instrument_type = searchScope['instrumentType']
    region = searchScope['region']
    delay = searchScope['delay']
    universe = searchScope['universe']

    if len(search) == 0:
        url_template = "https://api.worldquantbrain.com/data-fields?" + \
                       f"&instrumentType={instrument_type}" + \
                       f"&region={region}&delay={str(delay)}&universe={universe}&dataset.id={dataset_id}&limit=50" + \
                       "&offset={x}"
        count = s.get(url_template.format(x=0)).json()['count']
    else:
        url_template = "https://api.worldquantbrain.com/data-fields?" + \
                       f"&instrumentType={instrument_type}" + \
                       f"&region={region}&delay={str(delay)}&universe={universe}&limit=50" + \
                       f"&search={search}" + \
                       "&offset={x}"
        count = 100

    datafields_list = []
    for x in range(0, count, 50):
        datafields = s.get(url_template.format(x=x))
        datafields_list.append(datafields.json()['results'])

    datafields_list_flat = [item for sublist in datafields_list for item in sublist]

    datafields_df = pd.DataFrame(datafields_list_flat)
    return datafields_df

In [ ]:
# 定义搜索范围
searchScope = {'region': 'USA', 'delay': '1', 'universe': 'TOP3000', 'instrumentType': 'EQUITY'}
# 从数据集中获取数据字段
fnd6 = get_datafields(s=sess, searchScope=searchScope, dataset_id='fundamental6')

In [ ]:
# 将datafield和operator替换到Alpha模板(框架)中group_rank(ts_rank({fundamental model data},252),industry),批量生成Alpha
# 模板<group_compare_op>(<ts_compare_op>(<company_fundamentals>,<days>),<group>)
# 定义分组比较操作符
group_compare_op = ['group_rank', 'group_zscore', 'group_neutralize']  # 分组比较操作符列表
# 定义时间序列比较操作符
ts_compare_op = ['ts_rank', 'ts_zscore', 'ts_av_diff']  # 时间序列比较操作符列表
# 定义公司基本面数据的字段列表
company_fundamentals = datafields_list_fnd6
# 定义时间周期列表
days = [60, 200]
# 定义分组依据列表
group = ['market', 'industry', 'subindustry', 'sector', 'densify(pv13_h_f1_sector)']
# 初始化alpha表达式列表
alpha_expressions = []
# 遍历分组比较操作符
for gco in group_compare_op:
    # 遍历时间序列比较操作符
    for tco in ts_compare_op:
        # 遍历公司基本面数据的字段
        for cf in company_fundamentals:
            # 遍历时间周期
            for d in days:
                # 遍历分组依据
                for grp in group:
                    # 生成alpha表达式并添加到列表中
                    alpha_expressions.append(f"{gco}({tco}({cf}, {d}), {grp})")

# 输出生成的alpha表达式总数 # 打印或返回结果字符串列表
print(f"there are total {len(alpha_expressions)} alpha expressions")

In [ ]:
# 将datafield替换到Alpha模板(框架)中group_rank({fundamental model data}/cap,subindustry)批量生成Alpha
alpha_list = []

for index, alpha_expression in enumerate(alpha_expressions, start=1):
    print(f"正在循环第 {index} 个元素")
    print("正在将如下alpha表达式与setting封装")
    simulation_data = {
        "type": "REGULAR",
        "settings": {
            "instrumentType": "EQUITY",
            "region": "USA",
            "universe": "TOP3000",
            "delay": 1,
            "decay": 0,
            "neutralization": "SUBINDUSTRY",
            "truncation": 0.01,
            "pasteurization": "ON",
            "unitHandling": "VERIFY",
            "nanHandling": "OFF",
            "language": "FASTEXPR",
            "visualization": False,
        },
        "regular": alpha_expression
    }
    alpha_list.append(simulation_data)
    print(f"there are {len(alpha_list)} Alphas to simulate")

# 输出
print(alpha_list[0])

In [ ]:
import csv

# Write the CSV file
with open('alpha_list_pending_simulated.csv', 'w', newline='') as csvfile:
    fieldnames = ['type', 'settings', 'regular']
    writer = csv.DictWriter(csvfile, fieldnames=fieldnames)
    writer.writeheader()
    for item in alpha_list:
        writer.writerow({
            'type': item['type'],
            'settings': item['settings'],
            'regular': item['regular']
        })